In [1]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

# <p align="center"> Tools to Edit Gemmi.Structures</p>

- ## append_res()

Append a residue to a specific Chain and Model of a structure

In [20]:
# load test pdbs and modules

import gemmi
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model, sele_Lig
from xaidar.data.molecModels import get_pdb_stats, flatten_pdb
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
lig_pdb = sele_pdb(ev2a_pdb, sele_model)
lig_pdb = sele_pdb(lig_pdb, sele_Lig)
lig_res = flatten_pdb( lig_pdb, "residue")


In [21]:
ev2a_prot = protein_processing(ev2a_pdb)
print(flatten_pdb(ev2a_prot, "residue")[2])
GLY = flatten_pdb(ev2a_prot, "residue")[1]
SER =flatten_pdb(ev2a_prot, "residue")[0]
ev2a_prot = protein_processing(ev2a_pdb)
ev2a_prot[0]["B"].add_residue( flatten_pdb(lig_pdb, "chain")[0].whole()[0], pos = 1)
print(flatten_pdb(ev2a_prot, "residue")[0])

9(ALA)
7(SER)


In [22]:
def append_res_to_chain( new_res_lst, prot_chain_tobealtered: gemmi.Chain) -> None:
    prot_chain_tobealtered.append_residues( new_res_lst, min_sep = 1 )
    return None

In [23]:
ev2a_prot = protein_processing(ev2a_pdb)
append_res_to_chain( [lig_res[0], GLY, SER,lig_res[0]], ev2a_prot[0]["B"] )
print( flatten_pdb(  ev2a_prot, "residue")[-5:])

[<gemmi.Residue 146(GLU) with 9 atoms>, <gemmi.Residue 390(LIG) with 14 atoms>, <gemmi.Residue 148(GLY) with 4 atoms>, <gemmi.Residue 147(SER) with 6 atoms>, <gemmi.Residue 390(LIG) with 14 atoms>]


- ## add_res_to_chain

In [5]:
# load test pdbs and modules

import gemmi
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model, sele_Lig
from xaidar.data.molecModels import get_pdb_stats, flatten_pdb
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
lig_pdb = sele_pdb(ev2a_pdb, sele_model)
lig_pdb = sele_pdb(lig_pdb, sele_Lig)
lig_res = flatten_pdb( lig_pdb, "residue")


In [7]:
ev2a_prot = protein_processing(ev2a_pdb)
print(flatten_pdb(ev2a_prot, "residue")[2])
GLY = flatten_pdb(ev2a_prot, "residue")[1]
SER =flatten_pdb(ev2a_prot, "residue")[0]
ev2a_prot = protein_processing(ev2a_pdb)
ev2a_prot[0]["B"].add_residue( flatten_pdb(lig_pdb, "chain")[0].whole()[0], pos = 1)
print(flatten_pdb(ev2a_prot, "residue")[0])

9(ALA)
7(SER)


In [8]:
def add_res_to_chain( new_res_lst, new_res_pos_lst, 
                                prot_chain_tobealtered: gemmi.Chain) -> None:
    if ( ( not isinstance( new_res_lst, list) ) or 
                                ( not isinstance( new_res_pos_lst, list)) ):
        raise ValueError( " new_res, new_res_pos must be lists even "
                                                    "for one element") 
    elif len( new_res_lst) != len( new_res_pos_lst):
        raise ValueError( " new_res, new_res_pos lists must be of same length")

    lst_for_neg_index = [ (num, res) for num, res in 
                                zip( new_res_pos_lst, new_res_lst) if num < 0 ]
    lst_for_neg_index = sorted( lst_for_neg_index, key= lambda x: abs(x[0]))    # Sort negative indexes to process them in correct order
    
    lst_for_pos_index = [ (num, res) for num, res in 
                                zip( new_res_pos_lst, new_res_lst) if num >= 0]
    lst_for_pos_index = sorted( lst_for_pos_index, key= lambda x: x[0])         # So that residues are added in correct order to the right positions

    sorted_pairs = lst_for_neg_index + lst_for_pos_index
    for new_res_pos, new_res  in sorted_pairs:
        if new_res_pos < 0:
            new_res_pos = new_res_pos % len( prot_chain_tobealtered)            # Allow to account for negative indexing
        res_nums = [ res.seqid.num for res in prot_chain_tobealtered]           # get original indexes of old residues
        new_res.seqid.num = res_nums[new_res_pos]                               # update residue to be added seqid.num before adding it 
        new_res_nums = res_nums[:new_res_pos] + [ num + 1 
                                            for num in res_nums[new_res_pos:]]  # get new indexes for old residues          
        for res, new_num in zip( prot_chain_tobealtered, new_res_nums):         # Update old residues with new indexes
            res.seqid.num = new_num
        prot_chain_tobealtered.add_residue(new_res, new_res_pos)                # Add new residue to chain
    return None



In [17]:
ev2a_prot = protein_processing(ev2a_pdb)
add_res_to_chain( [lig_res[0], GLY, SER,lig_res[0]], [ -2,-1, -3, 0 ], ev2a_prot[0]["B"] )
print( flatten_pdb(  ev2a_prot, "residue"))

[<gemmi.Residue 7(LIG) with 14 atoms>, <gemmi.Residue 8(SER) with 6 atoms>, <gemmi.Residue 9(GLY) with 4 atoms>, <gemmi.Residue 10(ALA) with 5 atoms>, <gemmi.Residue 11(ILE) with 8 atoms>, <gemmi.Residue 12(TYR) with 12 atoms>, <gemmi.Residue 13(VAL) with 7 atoms>, <gemmi.Residue 14(GLY) with 4 atoms>, <gemmi.Residue 15(ASN) with 8 atoms>, <gemmi.Residue 16(TYR) with 12 atoms>, <gemmi.Residue 17(ARG) with 11 atoms>, <gemmi.Residue 18(VAL) with 7 atoms>, <gemmi.Residue 19(VAL) with 7 atoms>, <gemmi.Residue 20(ASN) with 8 atoms>, <gemmi.Residue 21(ARG) with 11 atoms>, <gemmi.Residue 22(HIS) with 10 atoms>, <gemmi.Residue 23(LEU) with 8 atoms>, <gemmi.Residue 24(ALA) with 5 atoms>, <gemmi.Residue 25(THR) with 7 atoms>, <gemmi.Residue 26(HIS) with 10 atoms>, <gemmi.Residue 27(ASN) with 8 atoms>, <gemmi.Residue 28(ASP) with 8 atoms>, <gemmi.Residue 29(TRP) with 14 atoms>, <gemmi.Residue 30(ALA) with 5 atoms>, <gemmi.Residue 31(ASN) with 8 atoms>, <gemmi.Residue 32(LEU) with 8 atoms>, <gemmi

- ## resList_to_resSpan

In [ ]:
# load test pdbs and modules
import gemmi
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import createPDB, flatten_pdb
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
ev2a_prot = protein_processing(ev2a_pdb)

In [ ]:
def resList_to_resSpan( res_lst: list) -> gemmi.ResidueSpan:
    pdb = createPDB(resSpan = res_lst)
    return flatten_pdb( pdb, "chain")[0].whole()

In [ ]:
print( resList_to_resSpan( flatten_pdb( ev2a_prot, "residue") ))

<gemmi.ResidueSpan of 140: Bxp [7(SER) 8(GLY) 9(ALA) ... 146(GLU)]>
